<div dir="rtl">
<h1>جهت مشتق را نگه دارید، اندازه‌اش را محدود کنید</h1>
<p>درس 55 از 76 · وقتی گام‌ها خیلی بزرگ یا کوچک‌اند · <code dir="ltr">49-rate</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-08/chapter-01/49-rate.html">📖 بازگشت به همین درس</a></p>
<p>محدودسازی Norm را از بریدن جداگانهٔ مؤلفه‌ها و از اندازهٔ نهایی update جدا کنید.</p><p>پیش‌نیاز: بردار، Norm و ترتیب backward تا optimizer.step.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>بردار [3,4] با سقف Norm برابر ۱، به [1,1] تبدیل می‌شود یا برداری هم‌جهت با خودش؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import torch
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
torch.set_num_threads(1)
torch.manual_seed(7)
model = MiniGPT(ModelConfig(12,4,8,2,1,0.0))
x,y = torch.tensor([[1,2,3]]),torch.tensor([[2,3,4]])
model(x,y)[1].backward()
norm = torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
print('norm before clipping:', norm.item())

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>clip_vector(Gradient, limit) برای بردار float و limit مثبت، یک بردار تازه با همان جهت و Norm حداکثر limit برگرداند. اگر Norm کوچک است یا بردار صفر است، مقدارها تغییر نکنند.</p>
</div>

In [ ]:
def clip_vector(gradient, limit):
    # TODO: یک ضریب مشترک برای کل بردار
    return None

In [ ]:
def test_exercise():
    source = torch.tensor([3.0,4.0])
    result = clip_vector(source,1.0)
    if result is None:
        return False
    torch.testing.assert_close(result,torch.tensor([0.6,0.8]))
    torch.testing.assert_close(source,torch.tensor([3.0,4.0]))
    torch.testing.assert_close(clip_vector(torch.tensor([0.1,0.2]),1.0),torch.tensor([0.1,0.2]))
    torch.testing.assert_close(clip_vector(torch.zeros(3),1.0),torch.zeros(3))
    torch.testing.assert_close(clip_vector(torch.tensor([-6.0,8.0]),2.0),torch.tensor([-1.2,1.6]))
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: clip_vector')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>Gradient ثابت است؛ فقط Learning Rate را عوض کنید. این مثال SGD است، نه فرمول کامل AdamW.</p>
</div>

In [ ]:
gradient = torch.tensor([0.6,0.8])
for rate in (0.01,0.1,1.0):
    update = -rate*gradient
    print(rate, 'update norm:', update.norm().item())

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>اگر ابتدا Step را انجام دهیم، محدودکردن مشتق دیگر آن تغییر وزن را اصلاح نمی‌کند. clipped_sgd(Value, Gradient, rate, limit) را برای عددهای Scalar بنویسید: اول Gradient را به بازهٔ مجاز ببرید و سپس وزن تازه را برگردانید.</p>
</div>

In [ ]:
value, gradient, rate, limit = 2.0, 10.0, 0.1, 1.0
wrong_value = value-rate*gradient
gradient = max(-limit,min(limit,gradient))
print('late clipping leaves weight at:',wrong_value)

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def clipped_sgd(value, gradient, rate, limit):
    # TODO: ترتیب محدودسازی و update
    return None

In [ ]:
def test_repair():
    result = clipped_sgd(2.0,10.0,0.1,1.0)
    if result is None:
        return False
    assert abs(result-1.9)<1e-8
    assert abs(clipped_sgd(2.0,-10.0,0.1,1.0)-2.1)<1e-8
    assert abs(clipped_sgd(2.0,0.5,0.1,1.0)-1.95)<1e-8
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: clipped_sgd')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>train.py مقدار بازگشتی clip_grad_norm_ را پیش از Step ثبت می‌کند. این مقدار Norm پیش از محدودسازی است، نه اندازهٔ تغییر وزن AdamW.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>اگر Norm محدود شد ولی Loss همچنان نامتناهی بود، چرا بزرگ‌ترکردن سقف راه‌حل قابل اتکایی نیست؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-08/chapter-01/49-rate.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/49-rate.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>